In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "2"

In [2]:
os.environ["HOME"] = "/mnt/nas/shuvranshu"
os.environ["HF_HOME"] = "/mnt/nas/shuvranshu/huggingface_cache"
os.environ["TRANSFORMERS_CACHE"] = "/mnt/nas/shuvranshu/huggingface_cache"
os.environ["HF_DATASETS_CACHE"] = "/mnt/nas/shuvranshu/huggingface_cache"
os.environ["XDG_CACHE_HOME"] = "/mnt/nas/shuvranshu/huggingface_cache"
os.environ["HF_DATASETS_CACHE"] = "/mnt/nas/shuvranshu/huggingface_cache"
os.makedirs("/mnt/nas/shuvranshu/huggingface_cache", exist_ok=True)

In [ ]:
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel,PeftConfig
import torch
import safetensors

/mnt/nas/shuvranshu/.conda/envs/finetune_env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
# ✅ 1. Paths
SAVE_DIR="/mnt/nas/shuvranshu/finetune"
base_model_path = "Qwen/Qwen2.5-3B-Instruct"
lora_adapter_path = f"{SAVE_DIR}/qwen_lora_adapter"

# ✅ 2. Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(base_model_path)

# ✅ 3. Load base model
base_model = AutoModelForCausalLM.from_pretrained(
    base_model_path,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True
)

# ✅ 4. Load LoRA adapter configuration
peft_config = PeftConfig.from_pretrained(lora_adapter_path)

# ✅ 5. Create a new PeftModel instance
model = PeftModel(base_model, peft_config)

# ✅ 6. Load the LoRA adapter weights using safetensors
adapter_weights = safetensors.torch.load_file(f"{lora_adapter_path}/model.safetensors")
model.load_state_dict(adapter_weights, strict=False)

print("✅ LoRA loaded successfully")

# ✅ 7. Move the model to GPU
model.to("cuda")

print("✅ Model loaded on GPU")

`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 434/434 [00:02<00:00, 205.86it/s]


✅ LoRA loaded successfully
✅ Model loaded on GPU


In [5]:

# Load the Bitext dataset
dataset = load_dataset("bitext/Bitext-customer-support-llm-chatbot-training-dataset")

# Define the category mapping
CATEGORY_MAP = {
    "ACCOUNT"          : "account",
    "BILLING"          : "billing",
    "CANCELLATION_FEE" : "billing",
    "REFUND"           : "billing",
    "PAYMENT"          : "billing",
    "DELIVERY"         : "shipping",
    "ORDER"            : "order",
    "TECHNICAL_SUPPORT": "technical",
    "CONTACT"          : "other",
    "FEEDBACK"         : "complaint",
    "COMPLAINT"        : "complaint",
}

# Apply the category mapping function
def map_category(row):
    raw = row["category"].upper().replace(" ", "_")
    for key in CATEGORY_MAP:
        if key in raw:
            row["mapped_category"] = CATEGORY_MAP[key]
            return row
    row["mapped_category"] = "other"
    return row

dataset = dataset.map(map_category)


In [7]:
print(dataset.keys())

dict_keys(['train'])


In [15]:
from datasets import load_dataset

# Load the dataset
# dataset = load_dataset("your_dataset_name")

# Split the "train" dataset into train and test subsets
train_test_split = dataset["train"].train_test_split(test_size=0.2, seed=42)
train_dataset = train_test_split["train"]
test_dataset = train_test_split["test"]

# Evaluate on the test subset
correct = 0
total = 0

SYSTEM_PROMPT = """You are a customer support query classifier.
Classify the user query into exactly one of these categories:
[billing, account, technical, shipping, order, complaint, other]
Respond with ONLY the category name, nothing else."""

for example in test_dataset:
    query = example["instruction"]
    true_category = example["mapped_category"]

    # Tokenize the query
    input_text = f"<|im_start|>system\n{SYSTEM_PROMPT}<|im_end|>\n<|im_start|>user\n{query}<|im_end|>\n<|im_start|>assistant\n"
    input_ids = tokenizer.encode(input_text, return_tensors="pt").to("cuda")

    # Generate the predicted category
    with torch.no_grad():
        output = model.generate(input_ids, max_new_tokens=5)

    generated_text = tokenizer.decode(output[0], skip_special_tokens=True).strip().lower()

    predicted_category = generated_text.split("assistant")[-1].strip()

    print(f"Question: {query}")
    print(f"Actual Category: {true_category}")
    print(f"Predicted Category: {predicted_category}")
    print("-------------------------------------")
    # Update accuracy metrics
    total += 1
    if predicted_category == true_category:
        correct += 1

accuracy = correct / total
print(f"Accuracy: {accuracy:.4f}")

Question: what do i need to do to change to the gold account
Actual Category: account
Predicted Category: account
-------------------------------------
Question: i try to modify the shippig address
Actual Category: other
Predicted Category: shipping
-------------------------------------
Question: how canI change order {{Order Number}}?
Actual Category: order
Predicted Category: order
-------------------------------------
Question: how could i enter afucking delivery address
Actual Category: other
Predicted Category: shipping
-------------------------------------
Question: can  uhelp me canceling order {{Order Number}}
Actual Category: order
Predicted Category: order
-------------------------------------
Question: I need assistance to set the delivery address up
Actual Category: other
Predicted Category: shipping
-------------------------------------
Question: where to download invoice #00108?
Actual Category: other
Predicted Category: billing
-------------------------------------
Quest